# Demo 1: Image Classification

**Before you start: go to Runtime → Change runtime type and select GPU.**

Image classification is the task of looking at an image and answering: 
*what is the main subject of this image?*

It sounds simple, but doing it reliably — across thousands of categories, 
in varying lighting, from different angles — is a hard problem that took 
decades of research to solve. Deep learning cracked it in 2012, and today 
image classifiers are everywhere: photo apps that tag your pictures automatically, 
medical imaging systems that screen X-rays, quality control systems in factories.

In this demo you will:
1. Load a state-of-the-art classifier that was trained on 1.2 million images
2. Run it on a few example images and see its predictions
3. Run it on an image of your own choice
4. See how transfer learning lets you build a custom classifier in minutes

You do not need to understand how any of this works yet — that is what 
the lectures are for. For now, just explore and pay attention to what 
impresses you and what surprises you.

## Setup

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request
import json
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Loading a pretrained classifier

We are going to use **ResNet-50**, a deep neural network trained on **ImageNet** — 
a dataset of 1.2 million images across 1,000 categories. Training this model from 
scratch takes days on multiple GPUs. We are going to load the finished result 
in about 10 seconds.

The model has learned to recognise everything from goldfish to volcanoes to 
aircraft carriers — purely by looking at examples.

In [ ]:
# Load ResNet-50 pretrained on ImageNet
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model = model.to(device)
model.eval()   # evaluation mode — disables dropout etc.

# How many parameters does this model have?
n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Parameters: {n_params:,}')

# Download the ImageNet class labels
url = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
urllib.request.urlretrieve(url, 'imagenet_labels.json')
with open('imagenet_labels.json') as f:
    imagenet_labels = json.load(f)

print(f'Number of categories: {len(imagenet_labels)}')
print(f'Some example categories: {imagenet_labels[0]}, {imagenet_labels[100]}, {imagenet_labels[500]}')

In [ ]:
# Define the preprocessing transform
# This must match exactly how the model was trained
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def classify_image(image_path, top_k=5):
    """
    Classify an image and return the top-k predictions.
    
    Args:
        image_path: path to image file or URL
        top_k:      number of top predictions to return
    """
    # Load image
    if image_path.startswith('http'):
        urllib.request.urlretrieve(image_path, 'temp_image.jpg')
        image_path = 'temp_image.jpg'
    image = Image.open(image_path).convert('RGB')
    
    # Preprocess
    tensor = preprocess(image).unsqueeze(0).to(device)  # add batch dimension
    
    # Run the model
    with torch.no_grad():
        logits = model(tensor)                          # raw scores
        probs  = F.softmax(logits, dim=1)[0]            # convert to probabilities
    
    # Get top-k predictions
    top_probs, top_indices = torch.topk(probs, top_k)
    predictions = [
        (imagenet_labels[idx.item()], prob.item())
        for idx, prob in zip(top_indices, top_probs)
    ]
    return image, predictions


def show_predictions(image, predictions, title=None):
    """Display an image alongside its top predictions."""
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Show image
    ax_img.imshow(image)
    ax_img.axis('off')
    if title:
        ax_img.set_title(title, fontsize=12)
    
    # Show predictions as a horizontal bar chart
    labels = [p[0] for p in predictions]
    probs  = [p[1] * 100 for p in predictions]   # convert to percentage
    colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(labels))]
    
    bars = ax_bar.barh(labels[::-1], probs[::-1], color=colors[::-1])
    ax_bar.set_xlabel('Confidence (%)')
    ax_bar.set_title('Top predictions')
    ax_bar.set_xlim(0, 100)
    
    # Add percentage labels on bars
    for bar, prob in zip(bars, probs[::-1]):
        ax_bar.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                   f'{prob:.1f}%', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()

print('Functions defined. Ready to classify images.')

## 2. Classifying example images

Let's run the model on a few images and see what it predicts.

In [ ]:
# A golden retriever
image, predictions = classify_image(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_Retriever_Dukedestiny01_drvd.jpg/320px-Golden_Retriever_Dukedestiny01_drvd.jpg'
)
show_predictions(image, predictions, title='What does the model see?')
print('Top prediction:', predictions[0][0], f'({predictions[0][1]*100:.1f}% confident)')

In [ ]:
# Something more ambiguous
image, predictions = classify_image(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/320px-Dog_Breeds.jpg'
)
show_predictions(image, predictions, title='Multiple objects — what does it focus on?')

In [ ]:
# Something that might challenge the model
image, predictions = classify_image(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg'
)
show_predictions(image, predictions, title='A close-up ant')

## 3. Try it on your own image

This is the interesting part. Find an image you are curious about — 
something from your project idea, something unusual, or something you 
think might confuse the model.

**Option A:** Paste an image URL directly:

In [ ]:
# Replace this URL with any image you want to try
my_image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg'

image, predictions = classify_image(my_image_url)
show_predictions(image, predictions)

**Option B:** Upload an image from your computer:

In [ ]:
# Run this cell to upload a file from your computer
from google.colab import files
uploaded = files.upload()

# Classify the uploaded file
for filename in uploaded.keys():
    image, predictions = classify_image(filename)
    show_predictions(image, predictions, title=filename)

## 4. Exploring the model's confidence

The model does not just output a single label — it outputs a probability 
for every one of the 1,000 categories. Let's look at the full distribution 
to get a sense of how certain or uncertain the model is.

In [ ]:
def show_confidence_distribution(image_path):
    """Show the full probability distribution across all 1000 classes."""
    if image_path.startswith('http'):
        urllib.request.urlretrieve(image_path, 'temp_image.jpg')
        image_path = 'temp_image.jpg'
    image = Image.open(image_path).convert('RGB')
    tensor = preprocess(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)[0].cpu().numpy()
    
    # Sort probabilities
    sorted_probs = np.sort(probs)[::-1]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1.imshow(image)
    ax1.axis('off')
    ax1.set_title('Input image')
    
    ax2.plot(sorted_probs * 100)
    ax2.set_xlabel('Rank (1 = most likely class)')
    ax2.set_ylabel('Probability (%)')
    ax2.set_title('Full probability distribution across 1,000 classes')
    ax2.set_xlim(0, 50)   # zoom in on the top 50
    ax2.grid(True, alpha=0.3)
    
    # Annotate top 3
    top3 = np.argsort(probs)[::-1][:3]
    for rank, idx in enumerate(top3):
        ax2.annotate(
            imagenet_labels[idx],
            xy=(rank, probs[idx] * 100),
            xytext=(rank + 2, probs[idx] * 100 + 2),
            fontsize=9,
            arrowprops=dict(arrowstyle='->', color='gray')
        )
    
    plt.tight_layout()
    plt.show()
    
    # How much probability mass is in the top-1, top-5, top-10?
    print(f'Top-1  confidence: {sorted_probs[0]*100:.1f}%')
    print(f'Top-5  confidence: {sorted_probs[:5].sum()*100:.1f}%')
    print(f'Top-10 confidence: {sorted_probs[:10].sum()*100:.1f}%')

# Try it on the golden retriever
show_confidence_distribution(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_Retriever_Dukedestiny01_drvd.jpg/320px-Golden_Retriever_Dukedestiny01_drvd.jpg'
)

## 5. Transfer learning: building a custom classifier in minutes

ResNet-50 knows 1,000 ImageNet categories. But what if you want to classify 
something it was never trained on — say, different species of mushrooms, 
or types of skin lesions, or stages of plant disease?

**Transfer learning** lets you reuse everything the model has already learned 
and adapt it to your own task. You replace the final layer (which outputs 1,000 
class scores) with a new layer for your number of classes, then train only that 
new layer on your own data.

The key insight: the model has already learned to recognise edges, textures, 
shapes, and object parts from 1.2 million images. That knowledge transfers 
to almost any visual task. You are building on top of years of prior work, 
not starting from scratch.

Let's see what this looks like in code:

In [ ]:
import torch.nn as nn

def build_custom_classifier(num_classes, freeze_backbone=True):
    """
    Adapt ResNet-50 for a custom classification task.
    
    The backbone (all layers except the last) keeps its pretrained weights.
    Only the final layer is replaced and trained from scratch.
    """
    # Load pretrained ResNet-50
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    
    if freeze_backbone:
        # Freeze all parameters — backbone weights will not change during training
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace the final layer with one for our number of classes
    in_features = model.fc.in_features   # 2048 for ResNet-50
    model.fc = nn.Linear(in_features, num_classes)
    # This new layer has requires_grad=True by default
    
    return model


# Example: a 5-class plant disease classifier
plant_model = build_custom_classifier(num_classes=5, freeze_backbone=True)

# How many parameters need to be trained?
trainable = sum(p.numel() for p in plant_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in plant_model.parameters())

print(f'Total parameters:     {total:,}')
print(f'Trainable parameters: {trainable:,}  ({trainable/total*100:.1f}% of the model)')
print()
print('With frozen backbone, you only need to train the final layer.')
print('This means you can get good results with much less data and training time.')

## 6. What can you do with this?

Image classification with transfer learning is the starting point for a huge 
range of real-world applications. Here are some examples of what students 
have built in this course before:

- **Medical imaging** — classifying skin lesions as benign or malignant
- **Wildlife monitoring** — identifying animal species from camera trap images
- **Agriculture** — detecting plant diseases from leaf photographs
- **Recycling** — classifying waste materials for automated sorting
- **Art** — classifying painting styles or artistic periods
- **Food** — classifying dishes from restaurant photos

The common thread: you have images, you have categories, and you want a model 
that can tell them apart. Transfer learning makes this achievable even with 
a modest dataset (a few hundred images per class can be enough).

---

### Think about your project

Take a few minutes with your group to discuss:

1. Is there a classification problem you find interesting? What would the classes be?
2. Do you know where you would find training data for it?
3. What would a correct prediction look like, and what would a wrong one cost?

Write down any ideas — you will use them when scoping your project proposal in the coming weeks.